# Building Agents with LangGraph Course #5: Persistence & Streaming in LangGraph

Companion notebook for [the complete To Data & Beyond tutorial](https://todatabeyond.com/blog/building-agents-with-langgraph-course-5-persistence-and-streaming-in-langgraph). View the [maintained notebook on GitHub](https://github.com/To-Data-Beyond/Generative-AI-Techanical-Tutorials/blob/main/LangGraph_Course_5_Persistence_and_Streaming.ipynb).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/To-Data-Beyond/Generative-AI-Techanical-Tutorials/blob/main/LangGraph_Course_5_Persistence_and_Streaming.ipynb)


## Before you begin

Add provider keys through Colab secrets or environment variables. Run cells in order; the lesson builds on graph state created in earlier cells.

This notebook intentionally contains no saved execution outputs or credentials.


## Environment setup


In [ ]:
!pip install -q langgraph langchain-openai langchain-community tavily-python python-dotenv langgraph-checkpoint-sqlite aiosqlite


When you start building complex AI agents designed for more than just a single turn of conversation, you’ll quickly run into two essential concepts: persistence and streaming. These features are the bedrock of creating robust, production-ready applications that can handle long-running tasks, maintain context, and provide a transparent user experience.

Persistence is the ability to save the state of your agent at any point in time. This allows you to pause and resume conversations, giving your agent a form of memory. It’s crucial for multi-turn dialogues and for enabling more advanced features like human-in-the-loop interventions.

Streaming provides a real-time view into your agent’s operations. Instead of waiting for a final answer, you can see the intermediate steps — like which tools are being called — or even the final answer as it’s being generated, token by token.

In this hands-on tutorial, the fifth article in the Building Agents with LangGraph series, we’ll take a basic LangGraph agent and enhance it with these powerful capabilities.

## Table of Contents

---

## 1. Setting Up the Agent

First, let’s get our initial agent set up. This will be the same research assistant agent from previous examples, equipped with a search tool. We begin by loading our environment variables and making the necessary imports.


In [ ]:
from dotenv import load_dotenv
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults

_ = load_dotenv()


Next, we’ll initialize our search tool and define the structure of our agent’s state. The state will simply contain a list of messages that we’ll append to over time.


In [ ]:
tool = TavilySearchResults(max_results=2)


In [ ]:
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]


---

## 2. Adding Persistence with Checkpointers

To add persistence, LangGraph introduces the concept of a checkpointer. A checkpointer automatically saves a snapshot of the graph’s state after every step (i.e., after each node is executed). This allows the agent to pick up right where it left off in a future interaction.

For this guide, we’ll use SqliteSaver, a straightforward checkpointer that uses a SQLite database. By passing “:memory:”, we create a temporary in-memory database that is perfect for testing. For production, you could easily point this to a file-based database or use more robust checkpointers for Postgres or Redis.


In [ ]:
from langgraph.checkpoint.sqlite import SqliteSaver

memory = SqliteSaver.from_conn_string(":memory:")


Now, we need to modify our Agent class to accept this checkpointer when the graph is compiled. We’ll add a checkpointer parameter to the __init__ method and pass it to graph.compile().


In [ ]:
class Agent:
    def __init__(self, model, tools, checkpointer, system=""):
        self.system = system
        graph = StateGraph(AgentState)
        graph.add_node("llm", self.call_openai)
        graph.add_node("action", self.take_action)
        graph.add_conditional_edges("llm", self.exists_action, {True: "action", False: END})
        graph.add_edge("action", "llm")
        graph.set_entry_point("llm")
        # Pass the checkpointer to the compile method
        self.graph = graph.compile(checkpointer=checkpointer)
        self.tools = {t.name: t for t in tools}
        self.model = model.bind_tools(tools)

    def call_openai(self, state: AgentState):
        messages = state['messages']
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages
        message = self.model.invoke(messages)
        return {'messages': [message]}

    def exists_action(self, state: AgentState):
        result = state['messages'][-1]
        return len(result.tool_calls) > 0

    def take_action(self, state: AgentState):
        tool_calls = state['messages'][-1].tool_calls
        results = []
        for t in tool_calls:
            print(f"Calling: {t}")
            result = self.tools[t['name']].invoke(t['args'])
            results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))
        print("Back to the model!")
        return {'messages': results}


With our modified class, we can now instantiate our agent, passing in the memory object we created.


In [ ]:
prompt = """You are a smart research assistant. Use the search engine to look up information. \
You are allowed to make multiple calls (either together or in sequence). \
Only look up information when you are sure of what you want. \
If you need to look up some information before asking a follow up question, you are allowed to do that!
"""
model = ChatOpenAI(model="gpt-4o")
abot = Agent(model, [tool], system=prompt, checkpointer=memory)


---

## 3. Streaming Intermediate Steps

With our checkpointer in place, let’s see how to manage conversational history. To do this, we need to introduce the concept of a thread. A thread is a unique identifier for a conversation, allowing the checkpointer to manage multiple conversations independently.

We define a thread using a configuration dictionary with a configurable key, which contains a thread_id.


In [ ]:
messages = [HumanMessage(content="What is the weather in sf?")]
thread = {"configurable": {"thread_id": "1"}}


Instead of using invoke to run our graph, we’ll now use stream. This will return an iterator of events, showing us the updates to the agent’s state at each step.


In [ ]:
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v['messages'])


**Expected output**

```text
[AIMessage(content=’’, additional_kwargs={‘tool_calls’: [{‘id’: ‘call_bmfLa92f6oAIKN9KvXtqbKDz’, ‘function’: {‘arguments’: ‘{“query”:”current weather in San Francisco”}’, ‘name’: ‘tavily_search_results_json’}, ‘type’: ‘function’}]}, response_metadata={‘token_usage’: {‘completion_tokens’: 22, ‘prompt_tokens’: 151, ‘total_tokens’: 173, ‘prompt_tokens_details’: {‘cached_tokens’: 0, ‘audio_tokens’: 0}, ‘completion_tokens_details’: {‘reasoning_tokens’: 0, ‘audio_tokens’: 0, ‘accepted_prediction_tokens’: 0, ‘rejected_prediction_tokens’: 0}}, ‘model_name’: ‘gpt-4o’, ‘system_fingerprint’: ‘fp_831e067d82’, ‘finish_reason’: ‘tool_calls’, ‘logprobs’: None}, id=’run-a71b0607–105c-45d5–8ca4–52ad9872b511–0', tool_calls=[{‘name’: ‘tavily_search_results_json’, ‘args’: {‘query’: ‘current weather in San Francisco’}, ‘id’: ‘call_bmfLa92f6oAIKN9KvXtqbKDz’}])]
Calling: {‘name’: ‘tavily_search_results_json’, ‘args’: {‘query’: ‘current weather in San Francisco’}, ‘id’: ‘call_bmfLa92f6oAIKN9KvXtqbKDz’}
Back to the model!
[ToolMessage(content=’[{\’url\’: \’https://www.weather25.com/north-america/usa/california/san-francisco?page=month&month=August\', \’content\’: \’| November | 18° / 10° | 3 | 27 | 0 | 37 mm | Good | San Francisco in November |\\n| December | 14° / 8° | 4 | 27 | 0 | 55 mm | Good | San Francisco in December | […] | Month | Temperatures | Rainy Days | Dry Days | Snowy Days | Rainfall | Weather | More details |\\n| — — | — — | — — | — — | — — | — — | — — | — — |\\n| January | 15° / 7° | 4 | 27 | 0 | 63 mm | Good | San Francisco in January |\\n| February | 16° / 7° | 4 | 24 | 0 | 61 mm | Good | San Francisco in February |\\n| March | 17° / 8° | 5 | 26 | 0 | 62 mm | Good | San Francisco in March |\\n| April | 18° / 9° | 2 | 28 | 0 | 22 mm | Good | San Francisco in April | […] Partly cloudy\\nPartly cloudy\\nSunny\\nPartly cloudy\\nMist\\nSunny\\nPartly cloudy\\nSunny\\nPartly cloudy\\nSunny\\nSunny\\nMist\\nMist\\nMist\\nPatchy rain possible\\nPatchy rain possible\\nSunny\\nSunny\\nSunny\\nSunny\\nSunny\\nPatchy rain possible\\nSunny\\nSunny\\nPartly cloudy\\nSunny\\nPartly cloudy\\nSunny\\nSunny\\nSunny\\nSunny\\n\\n## Explore the weather in San Francisco in other months\\n\\n## San Francisco annual weather\’}, {\’url\’: \’https://weather.com/weather/today/l/USCA0987:1:US\', \’content\’: “Some clouds in the morning will give way to mainly sunny skies for the afternoon. High 74F. Winds W at 15 to 25 mph.\\n\\n## Night\\n\\nPartly cloudy skies during the evening. Fog developing overnight. Low 59F. Winds W at 10 to 20 mph.\\n\\n## Radar\\n\\n## Travel\\n\\n## We Love Our Critters\\n\\n## Summer And Your Skin\\n\\n## Home, Garage & Garden\\n\\n## Live Maps Tracker\\n\\n## Keeping You Healthy\\n\\n## Product Reviews & Deals\\n\\nundefined\\n\\nPrep For Hurricane Season With These Generator Sales\\n\\nundefined […] ## Recent Locations\\n\\n## Weather Forecasts\\n\\n## Radar & Maps\\n\\n## News & Media\\n\\n## Products & Account\\n\\n## Lifestyle\\n\\n### Specialty Forecasts\\n\\n# San Francisco, CA\\n\\n## Small Craft Advisory\\n\\n## Weather Today in San Francisco, CA\\n\\n6:23 am\\n\\n8:05 pm\\n\\n# Hourly Weather-San Francisco, CA\\n\\n## Now\\n\\nPartly Cloudy\\n\\n## 6 pm\\n\\nMostly Cloudy\\n\\n## 7 pm\\n\\nMostly Cloudy\\n\\n## 8 pm\\n\\nCloudy\\n\\nChart small gif\\n\\n## Don\’t Miss\\n\\n## Seasonal Hub\\n\\n# 10 Day Weather-San Francisco, CA\\n\\n## Tonight\\n\\n## Night […] The Weather Channel is the world\’s most accurate forecaster according to ForecastWatch, Global and Regional Weather Forecast Accuracy Overview, 2021–2024, commissioned by The Weather Company.\\n\\nWeather Channel\\n\\n© The Weather Company, LLC 2025”}]’, name=’tavily_search_results_json’, tool_call_id=’call_bmfLa92f6oAIKN9KvXtqbKDz’)]
[AIMessage(content=’The current weather in San Francisco is partly cloudy in the morning, transitioning to mainly sunny skies in the afternoon with a high of 74°F (23°C). Winds are coming from the west at 15 to 25 mph. In the evening, the skies will be partly cloudy, with fog developing overnight and a low of 59°F (15°C). Winds will be from the west at 10 to 20 mph.’, response_metadata={‘token_usage’: {‘completion_tokens’: 87, ‘prompt_tokens’: 931, ‘total_tokens’: 1018, ‘prompt_tokens_details’: {‘cached_tokens’: 0, ‘audio_tokens’: 0}, ‘completion_tokens_details’: {‘reasoning_tokens’: 0, ‘audio_tokens’: 0, ‘accepted_prediction_tokens’: 0, ‘rejected_prediction_tokens’: 0}}, ‘model_name’: ‘gpt-4o’, ‘system_fingerprint’: ‘fp_07871e2ad8’, ‘finish_reason’: ‘stop’, ‘logprobs’: None}, id=’run-fafff744–614e-4a0f-ac9d-52536e25d1b8–0')]
```


When we run this, we see a stream of three distinct messages:

1. An AIMessage where the model decides to call the search tool.
2. A ToolMessage containing the results from the Tavily search.
3. A final AIMessage with the synthesized answer to our question.

This gives us excellent visibility into the agent’s reasoning process.

Now, let’s leverage the persistence we’ve set up. We can ask a follow-up question, and because we use the same thread_id, the agent will have access to the previous messages.


In [ ]:
messages = [HumanMessage(content="What about in la?")]
thread = {"configurable": {"thread_id": "1"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)


**Expected output**

```text
{‘messages’: [AIMessage(content=’’, additional_kwargs={‘tool_calls’: [{‘id’: ‘call_slsLp9eGg4f9xGQeUBLaAz5d’, ‘function’: {‘arguments’: ‘{“query”:”current weather in Los Angeles”}’, ‘name’: ‘tavily_search_results_json’}, ‘type’: ‘function’}]}, response_metadata={‘token_usage’: {‘completion_tokens’: 22, ‘prompt_tokens’: 1030, ‘total_tokens’: 1052, ‘prompt_tokens_details’: {‘cached_tokens’: 0, ‘audio_tokens’: 0}, ‘completion_tokens_details’: {‘reasoning_tokens’: 0, ‘audio_tokens’: 0, ‘accepted_prediction_tokens’: 0, ‘rejected_prediction_tokens’: 0}}, ‘model_name’: ‘gpt-4o’, ‘system_fingerprint’: ‘fp_ff25b2783a’, ‘finish_reason’: ‘tool_calls’, ‘logprobs’: None}, id=’run-2605c156–78cc-4c2e-9ee5–65ebc2f138c0–0', tool_calls=[{‘name’: ‘tavily_search_results_json’, ‘args’: {‘query’: ‘current weather in Los Angeles’}, ‘id’: ‘call_slsLp9eGg4f9xGQeUBLaAz5d’}])]}
Calling: {‘name’: ‘tavily_search_results_json’, ‘args’: {‘query’: ‘current weather in Los Angeles’}, ‘id’: ‘call_slsLp9eGg4f9xGQeUBLaAz5d’}
Back to the model!
{‘messages’: [ToolMessage(content=’[{\’url\’: \’https://www.weather25.com/north-america/usa/california/los-angeles?page=month&month=August\', \’content\’: \’| November | 24° / 14° | 1 | 29 | 0 | 22 mm | Perfect | Los Angeles in November |\\n| December | 19° / 11° | 3 | 28 | 0 | 66 mm | Good | Los Angeles in December | […] | Month | Temperatures | Rainy Days | Dry Days | Snowy Days | Rainfall | Weather | More details |\\n| — — | — — | — — | — — | — — | — — | — — | — — |\\n| January | 21° / 11° | 3 | 28 | 0 | 58 mm | Good | Los Angeles in January |\\n| February | 20° / 11° | 2 | 27 | 0 | 56 mm | Good | Los Angeles in February |\\n| March | 22° / 12° | 2 | 29 | 0 | 34 mm | Good | Los Angeles in March |\\n| April | 24° / 13° | 1 | 29 | 0 | 18 mm | Perfect | Los Angeles in April | […] Sunny\\nSunny\\nSunny\\nSunny\\nSunny\\nSunny\\nSunny\\nSunny\\nSunny\\nSunny\\nSunny\\nPartly cloudy\\nPartly cloudy\\nPartly cloudy\\nPartly cloudy\\nSunny\\nSunny\\nSunny\\nSunny\\nSunny\\nSunny\\nSunny\\nSunny\\nSunny\\nSunny\\nSunny\\nSunny\\nSunny\\nSunny\\nSunny\\nSunny\\n\\n## Explore the weather in Los Angeles in other months\\n\\n## Los Angeles annual weather\’}, {\’url\’: \’https://weather.com/weather/today/l/Los+Angeles+CA?canonicalCityId=7d3c65d8b80674fb48647ddbc936bb8b\', \’content\’: “## Recent Locations\\n\\n## Weather Forecasts\\n\\n## Radar & Maps\\n\\n## News & Media\\n\\n## Products & Account\\n\\n## Lifestyle\\n\\n### Specialty Forecasts\\n\\n# Los Angeles, CA\\n\\n## Weather Today in Los Angeles, CA\\n\\n6:13 am\\n\\n7:42 pm\\n\\n# Hourly Weather-Los Angeles, CA\\n\\n## Now\\n\\nSunny\\n\\n## 6 pm\\n\\nSunny\\n\\n## 7 pm\\n\\nSunny\\n\\n## 8 pm\\n\\nClear\\n\\nChart small gif\\n\\n## Don\’t Miss\\n\\n## Seasonal Hub\\n\\n# 10 Day Weather-Los Angeles, CA\\n\\n## Tonight\\n\\n## Night […] A few passing clouds, otherwise generally clear. Low around 65F. Winds light and variable.\\n\\n## Wed 13\\n\\n## Day\\n\\nSome clouds in the morning will give way to mainly sunny skies for the afternoon. High 84F. Winds light and variable.\\n\\n## Night\\n\\nClear skies with a few passing clouds. Low 64F. Winds light and variable.\\n\\n## Thu 14\\n\\n## Day\\n\\nSome clouds in the morning will give way to mainly sunny skies for the afternoon. High 83F. Winds light and variable.\\n\\n## Night […] The Weather Channel is the world\’s most accurate forecaster according to ForecastWatch, Global and Regional Weather Forecast Accuracy Overview, 2021–2024, commissioned by The Weather Company.\\n\\nWeather Channel\\n\\n© The Weather Company, LLC 2025”}]’, name=’tavily_search_results_json’, tool_call_id=’call_slsLp9eGg4f9xGQeUBLaAz5d’)]}
{‘messages’: [AIMessage(content=’The current weather in Los Angeles is sunny, with a high of 84°F (29°C). Winds are light and variable. In the evening, the weather will be generally clear with a few passing clouds and a low around 65°F (18°C).’, response_metadata={‘token_usage’: {‘completion_tokens’: 53, ‘prompt_tokens’: 1796, ‘total_tokens’: 1849, ‘prompt_tokens_details’: {‘cached_tokens’: 0, ‘audio_tokens’: 0}, ‘completion_tokens_details’: {‘reasoning_tokens’: 0, ‘audio_tokens’: 0, ‘accepted_prediction_tokens’: 0, ‘rejected_prediction_tokens’: 0}}, ‘model_name’: ‘gpt-4o’, ‘system_fingerprint’: ‘fp_07871e2ad8’, ‘finish_reason’: ‘stop’, ‘logprobs’: None}, id=’run-36a5ea35–625c-4df3–986a-add2ef06dcda-0')]}
```


The agent correctly infers that we’re still asking about the weather and searches for the weather in Los Angeles. It remembers the context of our conversation.

Let’s ask one more question in the same thread:


In [ ]:
messages = [HumanMessage(content="Which one is warmer?")]
thread = {"configurable": {"thread_id": "1"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)


**Expected output**

```text
{‘messages’: [AIMessage(content=’Los Angeles is warmer than San Francisco. Los Angeles has a high of 84°F (29°C), while San Francisco has a high of 74°F (23°C).’, response_metadata={‘token_usage’: {‘completion_tokens’: 36, ‘prompt_tokens’: 1861, ‘total_tokens’: 1897, ‘prompt_tokens_details’: {‘cached_tokens’: 0, ‘audio_tokens’: 0}, ‘completion_tokens_details’: {‘reasoning_tokens’: 0, ‘audio_tokens’: 0, ‘accepted_prediction_tokens’: 0, ‘rejected_prediction_tokens’: 0}}, ‘model_name’: ‘gpt-4o’, ‘system_fingerprint’: ‘fp_ff25b2783a’, ‘finish_reason’: ‘stop’, ‘logprobs’: None}, id=’run-859be8d8–9028–4e16–8466–9658de637bec-0')]}
```


The agent uses the history of the entire conversation to determine that Los Angeles is warmer.

To prove the importance of the thread_id, let’s ask the same question but change the thread_id to “2”.


In [ ]:
messages = [HumanMessage(content="Which one is warmer?")]
thread = {"configurable": {"thread_id": "2"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)


**Expected output**

```text
{‘messages’: [AIMessage(content=”Could you please clarify what you’re comparing to determine which is warmer? Are you comparing two specific locations, types of clothing, materials, or something else? Let me know so I can provide the appropriate information.”, response_metadata={‘token_usage’: {‘completion_tokens’: 43, ‘prompt_tokens’: 149, ‘total_tokens’: 192, ‘prompt_tokens_details’: {‘cached_tokens’: 0, ‘audio_tokens’: 0}, ‘completion_tokens_details’: {‘reasoning_tokens’: 0, ‘audio_tokens’: 0, ‘accepted_prediction_tokens’: 0, ‘rejected_prediction_tokens’: 0}}, ‘model_name’: ‘gpt-4o’, ‘system_fingerprint’: ‘fp_f9f4fb6dbf’, ‘finish_reason’: ‘stop’, ‘logprobs’: None}, id=’run-3b304797–71ba-48bf-b2eb-7bb32977dc08–0')]}
```


The model is now confused, asking for clarification. Because we used a new thread ID, it has no access to the prior conversation history. This perfectly illustrates how checkpointers and threads enable stateful, multi-turn interactions.

---

## 4. Streaming Tokens in Real Time

Besides streaming the intermediate steps, we can also stream the final output token-by-token, just like you see in applications like ChatGPT. This requires using an asynchronous method: astream_events.

To use async methods on the graph, we first need to switch to an async checkpointer. The AsyncSqliteSaver is the async equivalent of the one we used before.


In [ ]:
from langgraph.checkpoint.aiosqlite import AsyncSqliteSaver

memory = AsyncSqliteSaver.from_conn_string(":memory:")
abot = Agent(model, [tool], system=prompt, checkpointer=memory)


Now, we can call astream_events. We’ll loop through the events and look for a specific kind: “on_chat_model_stream”. When we find one, we’ll print the content of the chunk. We add a check for content because empty chunks signify a tool call is being requested.


In [ ]:
messages = [HumanMessage(content="What is the weather in SF?")]
thread = {"configurable": {"thread_id": "4"}}
# The following is an async loop
async for event in abot.graph.astream_events({"messages": messages}, thread, version="v1"):
    kind = event["event"]
    if kind == "on_chat_model_stream":
        content = event["data"]["chunk"].content
        if content:
            # Empty content in the context of OpenAI means
            # that the model is asking for a tool to be invoked.
            # So we only print non-empty content
            print(content, end="|")


Running this code produces a real-time stream of tokens for the final answer, providing a much more interactive user experience.

![LangGraph agent response appearing token by token in real time](https://todatabeyond.com/articles/langgraph-persistence-streaming/token-streaming-demo.gif)

*Streaming the final LangGraph response token by token with astream_events.*

---

By incorporating persistence and streaming, you can elevate your LangGraph agents from simple, one-shot tools to sophisticated, stateful applications.

Persistence through checkpointers provides the memory needed for coherent, long-running conversations and is a key enabler of human-in-the-loop workflows, a topic we’ll explore next.

Streaming, both of intermediate steps and of final tokens, offers crucial transparency and a more dynamic user experience. These features are essential for building the next generation of powerful and reliable AI agents.
